# Day 14 — NumPy Statistical Functions
### Python for Data Science · Module 1 · Topic 1.13

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Centre — mean, median, and why they differ | 20 min |
| 2 | Spread — std, var, percentiles, IQR | 20 min |
| 3 | **Missing data — `nan` and the nan-aware functions** | 30 min |
| 4 | Relationships — correlation and covariance | 15 min |
| 5 | Mini build: an honest summary report | 5 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **A warning about this session.** Every function today returns a number, and a number
> always looks authoritative. An average computed over data you have not examined can be
> confidently, precisely wrong — one outlier, or one missing value handled carelessly, is
> enough. Today is as much about knowing which summary to trust as how to compute it.

In [ ]:
import numpy as np
print("numpy", np.__version__)

---
# 1. Centre

## 1.1 The mean is not always the average you want

In [ ]:
# Six salaries, in thousands
salaries = np.array([30., 32., 35., 33., 31., 500.])

print("mean   :", salaries.mean().round(2))
print("median :", np.median(salaries))
print()
print("Five people earn about 32. The mean describes nobody.")

In [ ]:
# Remove the outlier and watch the mean snap back
clean = np.array([30., 32., 35., 33., 31.])
print("without the outlier -> mean", clean.mean(), " median", np.median(clean))

# The mean uses EVERY value, so one extreme number drags it a long way.
# The median only cares about the middle POSITION, so it barely moves.

**Which one to use:**

- Symmetric data, no extremes → **mean**
- Skewed data, or outliers → **median**
- Report **both** when they disagree — the gap is itself information

## 1.2 Weighted average

In [ ]:
marks   = np.array([88., 71., 64.])
weights = np.array([0.5, 0.3, 0.2])

print("np.average :", np.average(marks, weights=weights))
print("by hand    :", (marks * weights).sum())

# np.average takes weights; np.mean does not.

## 1.3 There is no `np.mode`

In [ ]:
print("does np.mode exist?", hasattr(np, "mode"))

# NumPy is a numerical ARRAY library, not a statistics package. The mode is
# awkward to define - a dataset can have two modes, or none - so NumPy leaves
# it out rather than guessing what you want.

x = np.array([3, 1, 4, 1, 5, 1, 4])
vals, counts = np.unique(x, return_counts=True)

print("values :", vals)
print("counts :", counts)
print("mode   :", vals[counts.argmax()])      # Day 13's argmax

In [ ]:
# np.unique is worth knowing for its own sake
x = np.array([3, 1, 4, 1, 5, 1, 4])

print("sorted, duplicates gone:", np.unique(x))
print("how many distinct      :", len(np.unique(x)))

# This is Day 4's set, but sorted and countable. In two weeks pandas gives
# you value_counts(), which does exactly this and sorts by frequency.

---
# 2. Spread

## 2.1 Two datasets with the same mean

In [ ]:
a = np.array([48., 49., 50., 51., 52.])
b = np.array([10., 30., 50., 70., 90.])

print("means :", a.mean(), b.mean())        # identical
print("stds  :", a.std().round(2), b.std().round(2))   # not remotely
print()
print("var = std ** 2 :", a.var().round(2), "vs", (a.std()**2).round(2))
print("std is in the SAME UNITS as the data; var is in squared units.")

### ⚠️ NumPy is inconsistent about `ddof`

In [ ]:
arr = np.array([1., 2., 3., 4.])

print("arr.std()        :", arr.std().round(4), "  ddof=0, divides by n")
print("arr.std(ddof=1)  :", arr.std(ddof=1).round(4), "  divides by n-1")
print()
print("but np.cov defaults to ddof=1:")
print("  np.cov(arr)  :", np.cov(arr).round(4))
print("  arr.var(ddof=1):", arr.var(ddof=1).round(4), " <- they match")

# std and var default to the POPULATION formula.
# np.cov defaults to the SAMPLE formula. That is NumPy's own inconsistency.
# pandas defaults everything to ddof=1.

## 2.2 Percentiles and the IQR

In [ ]:
a = np.arange(1., 11.)      # 1 to 10
print("data:", a)
print()
for p in [0, 25, 50, 75, 100]:
    print(f"  p{p:<3} = {np.percentile(a, p)}")

print()
print("median == p50 :", np.median(a) == np.percentile(a, 50))
print("ptp (max-min) :", np.ptp(a))

In [ ]:
q1, q3 = np.percentile(a, [25, 75])
iqr = q3 - q1

print("Q1 :", q1)
print("Q3 :", q3)
print("IQR:", iqr, " <- the spread of the MIDDLE HALF")

# Outliers sit outside the middle half by definition, so they cannot
# distort the IQR the way they distort the range or the std.

### You will meet this again on Day 20

```python
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
```

Anything outside those bounds is the standard definition of an outlier — and it is exactly
what draws the whiskers on a box plot.

In [ ]:
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"outlier bounds: below {lower} or above {upper}")

test = np.array([1., 5., 9., 50.])
print("test data  :", test)
print("outliers   :", test[(test < lower) | (test > upper)])

---
# 3. Missing data

## 3.1 `nan` — the value that is not a value

Where it comes from:

- a blank cell in a CSV
- a survey question nobody answered
- a sensor that failed for an hour
- `0 / 0`, or the log of a negative number

In [ ]:
d = np.array([1., 2., np.nan, 4.])
print("array :", d)
print("dtype :", d.dtype, " <- always a float")

### ⚠️ `nan` is not equal to itself

In [ ]:
print("np.nan == np.nan :", np.nan == np.nan)     # False (!)
print("d == np.nan      :", d == np.nan)          # all False
print()
print("np.isnan(d)      :", np.isnan(d))          # the correct way

# "Unknown" cannot equal "unknown" - you do not know whether the two missing
# values were the same. So == never finds a nan. You must use np.isnan.

### ⚠️ One `nan` poisons the whole calculation

In [ ]:
d = np.array([1., 2., np.nan, 4.])

print("d.sum()  :", d.sum())
print("d.mean() :", d.mean())
print("d.max()  :", d.max())

# Any arithmetic touching a nan produces a nan, so ONE missing value turns an
# entire summary into nan. That is deliberate: NumPy refuses to quietly invent
# an answer. Your job is to decide what should happen instead.

## 3.2 Two ways to compute around it

In [ ]:
# 1. The nan-aware functions - each skips the nans
print("np.nanmean(d)   :", np.nanmean(d).round(4))
print("np.nanmedian(d) :", np.nanmedian(d))
print("np.nansum(d)    :", np.nansum(d))
print("np.nanstd(d)    :", np.nanstd(d).round(4))
print("np.nanmin/max   :", np.nanmin(d), np.nanmax(d))
print("np.nanpercentile:", np.nanpercentile(d, 50))

In [ ]:
# 2. Filter them out yourself
clean = d[~np.isnan(d)]          # ~ is 'not' - Day 12's boolean mask

print("clean       :", clean)
print("clean.mean():", clean.mean().round(4), " <- same answer")
print("dropped     :", np.isnan(d).sum(), "values")

# Prefer this when you want to REPORT how many values you dropped.

### ⚠️ Two edge cases that will surprise you

In [ ]:
import warnings

all_nan = np.array([np.nan, np.nan])

print("np.nansum(all_nan)  :", np.nansum(all_nan), "  (!)")
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print("np.nanmean(all_nan) :", np.nanmean(all_nan), " + a RuntimeWarning")

# A sum of NOTHING is defined as 0, so nansum reports 0.0 for a column that is
# entirely missing. A real zero and a missing column look IDENTICAL in your
# output.
#
# ALWAYS report the count alongside any nan-aware summary.

## 3.3 Finding and counting what is missing

In [ ]:
d = np.array([1., 2., np.nan, 4., np.nan])

print("how many missing :", np.isnan(d).sum())
print("any at all?      :", np.isnan(d).any())
print("the proportion   :", np.isnan(d).mean())     # True counts as 1

# isnan returns a BOOLEAN ARRAY, so every Day 12 and Day 13 trick applies.

In [ ]:
# Per column, on a 2-D array
m = np.array([[1., 2., np.nan],
              [4., np.nan, 6.],
              [7., 8., 9.]])

print("missing per column:", np.isnan(m).sum(axis=0))
print("missing per row   :", np.isnan(m).sum(axis=1))
print("column means      :", np.nanmean(m, axis=0).round(2))

### ⚠️ Integer arrays cannot hold `nan`

In [ ]:
print("dtype with a nan:", np.array([1, 2, np.nan]).dtype)

try:
    np.array([1, 2, np.nan], dtype=int)
except ValueError as e:
    print("forcing int ->", type(e).__name__, ":", str(e)[:45])

# nan is a FLOATING-POINT value - there is no integer bit pattern for
# "missing". So the moment a column contains one nan, the whole column
# becomes float.
#
# If your integer IDs suddenly print as 1.0 and 2.0, a missing value is why.

**Then decide — and say so:**

- **Drop the rows** — fine if few, dangerous if many
- **Fill with the mean or median** — keeps the row, distorts the spread
- **Leave them** and use nan-aware functions

Whichever you choose, record how many there were.

---
# 4. Relationships

In [ ]:
hours  = np.array([1., 2., 3., 4., 5.])
scores = np.array([52., 55., 61., 68., 74.])

print(np.corrcoef(hours, scores).round(4))
print()
print("corrcoef returns a MATRIX, not a number.")
print("The diagonal is always 1 - every variable correlates with itself.")
print()
print("r =", np.corrcoef(hours, scores)[0, 1].round(4))

| `r` | Means |
|---|---|
| `+1` | perfect: they go up together |
| `+0.7` | strong positive |
| `0` | no **linear** relation |
| `-0.7` | strong negative |
| `-1` | perfect: one up, one down |

In [ ]:
# The extremes, to anchor the scale
print("perfectly positive:", np.corrcoef(hours, hours * 2 + 1)[0, 1].round(6))
print("perfectly negative:", np.corrcoef(hours, -hours)[0, 1].round(6))

# Covariance is the unscaled version - same sign, but its size depends on
# the units, which is why correlation is easier to interpret.
print()
print("covariance matrix:")
print(np.cov(hours, scores).round(3))

### ⚠️ Two things `r` does **not** tell you

**It measures only straight-line relationships.** A perfect U-shaped curve can give `r` close
to 0 — the relationship is real and strong, just not linear. Always plot before you trust a
number.

**It says nothing about cause.** Ice cream sales correlate with drowning deaths; neither
causes the other, because summer causes both. `r = 0.99` is a reason to investigate, never a
conclusion.

In [ ]:
# A perfect relationship that correlation completely misses
x = np.array([-3., -2., -1., 0., 1., 2., 3.])
y = x ** 2                      # a perfect parabola

print("y is EXACTLY determined by x")
print("but r =", np.corrcoef(x, y)[0, 1].round(6))
print()
print("Correlation found nothing, because the relationship is not a straight line.")

---
# 5. Putting it together — an honest summary report

In [ ]:
# marks for 6 students - two are missing
marks = np.array([88., 71., np.nan, 64., 95., np.nan])


def describe(a, name="data"):
    """Summarise an array honestly, missing values included."""
    missing = np.isnan(a)
    n_missing = missing.sum()
    clean = a[~missing]

    if clean.size == 0:                       # everything was missing
        print(f"{name}: no usable values")
        return

    q1, q3 = np.percentile(clean, [25, 75])
    print(f"{name}")
    print(f"  count   {clean.size} of {a.size}   ({n_missing} missing)")
    print(f"  mean    {clean.mean():.2f}")
    print(f"  median  {np.median(clean):.2f}")
    print(f"  std     {clean.std(ddof=1):.2f}")
    print(f"  range   {clean.min():.0f} to {clean.max():.0f}")
    print(f"  IQR     {q3 - q1:.2f}")


describe(marks, "marks")
print()
describe(np.array([np.nan, np.nan]), "all missing")

- **Count first** — how much of the data is real
- **Mask once** — `~missing` reused for everything
- **Guard the empty case** — an all-missing array does not crash
- **mean AND median** — the gap reveals skew
- **`ddof=1`** — the sample std, stated explicitly
- **IQR too** — spread an outlier cannot distort

> Reporting the count is what makes this honest. A mean over 4 of 6 values is a different
> claim from a mean over all 6.

In two weeks pandas gives you `.describe()`, which does most of this — but it does **not**
print the count of missing values by default, which is exactly why knowing what a summary
should contain matters.

---
# 6. Recap — the twelve things to remember

1. The mean uses every value; one outlier drags it a long way.
2. The median barely moves. Report both when they disagree.
3. `np.average` takes weights; `np.mean` does not.
4. There is no `np.mode` — build it from `np.unique`.
5. `std` and `var` default to `ddof=0`; `np.cov` defaults to `ddof=1`.
6. `np.percentile(a, 50)` is the median. `IQR = Q3 - Q1`.
7. `nan != nan`, so `== nan` never matches. Use `np.isnan`.
8. One `nan` makes any sum, mean or max return `nan`.
9. `np.nanmean` and friends skip them; so does `d[~np.isnan(d)]`.
10. `np.nansum` of an all-missing array returns `0.0`, not `nan`.
11. A `nan` forces the whole array to float — ints cannot hold it.
12. `corrcoef` returns a **matrix**; take `[0, 1]`. `r` is not cause.

---

### 📝 Now open **`Day14_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Write `describe()` for a 2-D array, reporting per column.
- Find the outliers in a dataset using the 1.5 × IQR rule.
- Take a clean array, blank out three values, and compare every summary before and after.

### Next class — Topic 1.14: pandas Series & DataFrames
The labelled table that everything from here runs on. NumPy underneath, names on top.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*